# 🎩 Butler — GRPO Training Notebook

Train a Butler personal task orchestration agent using **GRPO** (Group Relative Policy Optimization)
via **Unsloth + HF TRL** on Google Colab.

## What this notebook does:
1. Installs dependencies
2. Loads the Butler OpenEnv environment (configured for HF or Cursor APIs)
3. Generates synthetic training data
4. Fine-tunes Qwen2.5-7B-Instruct with GRPO
5. Plots reward curves and rubric breakdowns
6. Compares baseline vs trained agent
7. Pushes trained model to HF Hub

**Note:** To run inference properly, you will need either a HuggingFace Token (`HF_TOKEN`) or a Cursor API Key (`CURSOR_API_KEY`) set in your environment.

## Cell 1: Install Dependencies

In [ ]:
%%capture
!pip install -q openenv unsloth transformers trl datasets wandb \
    google-api-python-client google-auth-httplib2 \
    google-auth-oauthlib matplotlib huggingface-hub

## Cell 2: Configuration (Fill in your credentials)

In [ ]:
# ═══════════════════════════════════════════════
# USER CONFIG — Fill these in before running
# ═══════════════════════════════════════════════

HF_TOKEN        = ""    # Your Hugging Face token
CURSOR_API_KEY  = ""    # Optional: Your Cursor API key (fallback LLM)
WANDB_API_KEY   = ""    # Your Weights & Biases key
BASE_MODEL      = "unsloth/Qwen2.5-7B-Instruct"
OUTPUT_MODEL    = "your-hf-username/butler-grpo"
MAX_STEPS       = 150
BATCH_SIZE      = 4
LR              = 2e-5
SAVE_STEPS      = 100
EVAL_STEPS      = 50


## Cell 3: Auth + WandB Init

In [ ]:
import wandb
import os
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ["HF_TOKEN"] = HF_TOKEN

if CURSOR_API_KEY:
    os.environ["CURSOR_API_KEY"] = CURSOR_API_KEY

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    wandb.init(project="butler-openenv", name="butler-grpo-run-1")
    print("✅ WandB initialized.")

print("✅ Auth complete.")


## Cell 4: Clone & Load Butler Environment

In [ ]:
# Load the Butler Environment Files
import os
import sys

if not os.path.exists('env'):
    print("Butler environment files not found.")
    print("Please upload the 'butler-openenv' folder to this Colab session.")
    print("You can either drag and drop the folder, or zip it on your computer, upload it, and run:")
    print("!unzip butler-openenv.zip")
    # Alternatively, if you pushed to GitHub/HF, uncomment the line below:
    # !git clone https://huggingface.co/spaces/YOUR-USERNAME/YOUR-SPACE-NAME .

sys.path.insert(0, '.')

try:
    from env.butler_env import ButlerEnvironment
    from env.observation import build_observation_prompt, SYSTEM_PROMPT
    from env.action_space import parse_llm_output, validate_action
    from agents.orchestrator import Orchestrator

    # Verify environment
    env = ButlerEnvironment()
    obs = env.reset()
    print(f"Queue size: {len(obs['queue'])}")
    print(f"Current todo: {obs['current_todo']['text']}")
    print(f"Tier: {obs['current_todo']['tier']}")
    
    # Verify priority sorting
    tiers = [t['tier'] for t in obs['queue']]
    print(f"Queue order (should be TIER1 first): {tiers}")
except ModuleNotFoundError:
    print("❌ ERROR: The Butler files are missing. You must upload them to Colab first.")


## Cell 5: Generate Synthetic Dataset

In [ ]:
from data.synthetic_todos import save_dataset, generate_batch
from datasets import Dataset
import json

# Generate and save dataset
save_dataset(
    path="data/butler_dataset.json",
    n_train=500,
    n_eval=100,
    n_test=100
)

# Load as HF Dataset
with open("data/butler_dataset.json") as f:
    raw = json.load(f)

# Flatten episodes for Dataset format
train_data = [{"episode": ep} for ep in raw["train"]]
eval_data = [{"episode": ep} for ep in raw["eval"]]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print(f"Train: {len(train_dataset)} episodes")
print(f"Eval:  {len(eval_dataset)} episodes")

## Cell 6: Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"✅ Model loaded: {BASE_MODEL}")
print(f"   Trainable params: {model.print_trainable_parameters()}")

## Cell 7: Define Reward Function

In [ ]:
from env.butler_env import ButlerEnvironment
from env.action_space import parse_llm_output, validate_action

# Create a fresh env for reward computation
reward_env = ButlerEnvironment()


def butler_reward_fn(completions, prompts, **kwargs):
    """
    Reward function for GRPO training.
    
    For each completion string:
    1. Parse JSON action from completion
    2. Validate action schema
    3. env.step(action) to get reward
    4. Return reward float (0.0 on parse/validation failure)
    """
    rewards = []
    for completion in completions:
        try:
            action = parse_llm_output(completion)
            if action is None:
                rewards.append(0.0)
                continue
            
            valid, error = validate_action(action)
            if not valid:
                rewards.append(0.1)  # Small reward for parseable but invalid
                continue
            
            # Reset env with a quick episode for scoring
            reward_env.reset()
            _, reward, _, info = reward_env.step(action)
            rewards.append(float(reward))
            
        except Exception as e:
            rewards.append(0.0)
    
    return rewards


print("✅ Reward function defined.")

## Cell 8: Format Dataset for GRPO

In [ ]:
from env.observation import build_observation_prompt, SYSTEM_PROMPT
from agents.orchestrator import Orchestrator

orch = Orchestrator()


def format_episode(example):
    """
    Convert an episode queue (list of todos) into a prompt for GRPO.
    """
    episode_queue = example["episode"]
    sorted_queue = orch.sort_queue(episode_queue)
    
    obs = {
        "queue": sorted_queue,
        "current_todo": sorted_queue[0] if sorted_queue else None,
        "user_context": {
            "name": "User",
            "timezone": "Asia/Kolkata",
            "communication_style": "professional",
            "role": "Professional",
        },
        "step": 0,
        "max_steps": 10,
    }
    
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{build_observation_prompt(obs)}\n"
        f"<|assistant|>\n"
    )
    
    return {"prompt": prompt}


train_dataset = train_dataset.map(
    format_episode,
    remove_columns=train_dataset.column_names,
)

eval_dataset = eval_dataset.map(
    format_episode,
    remove_columns=eval_dataset.column_names,
)

print(f"✅ Dataset formatted.")
print(f"   Train prompts: {len(train_dataset)}")
print(f"   Eval prompts:  {len(eval_dataset)}")
print(f"\n   Sample prompt (first 500 chars):")
print(train_dataset[0]['prompt'][:500])

## Cell 9: GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./butler-grpo-output",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    max_steps=MAX_STEPS,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    report_to="wandb",
    run_name="butler-grpo-run-1",
    max_completion_length=256,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=butler_reward_fn,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("🚀 Starting GRPO training...")
trainer.train()
print("✅ Training complete!")

## Cell 10: Plot Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("assets", exist_ok=True)

log_history = trainer.state.log_history

steps   = [x["step"]   for x in log_history if "reward" in x]
rewards = [x["reward"] for x in log_history if "reward" in x]

plt.figure(figsize=(10, 5))
plt.plot(steps, rewards, color="#6C63FF", linewidth=2, label="Butler GRPO")
plt.axhline(
    y=0.21, color="#FF6B6B", linestyle="--",
    linewidth=1.5, label="Random baseline (0.21)"
)
plt.xlabel("Training Step")
plt.ylabel("Reward (0–1)")
plt.title("Butler GRPO Training — Reward vs Step")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("assets/reward_curve.png", dpi=150)
plt.show()
print("Saved: assets/reward_curve.png")

## Cell 11: Per-Rubric Breakdown Plot

In [ ]:
rubric_keys = [
    "priority_ordering", "correct_routing",
    "action_completeness", "api_call_success",
    "no_over_triggering"
]
colors = ["#6C63FF", "#4ECDC4", "#F7B731", "#FC5C65", "#45AAF2"]

plt.figure(figsize=(12, 5))
for key, color in zip(rubric_keys, colors):
    vals = [x.get(key, None) for x in log_history if key in x]
    s    = [x["step"]        for x in log_history if key in x]
    if vals:
        plt.plot(
            s, vals,
            label=key.replace("_", " ").title(),
            color=color, linewidth=1.5
        )

plt.xlabel("Training Step")
plt.ylabel("Rubric Score (0–1)")
plt.title("Per-Rubric Component Scores During Training")
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("assets/rubric_breakdown.png", dpi=150)
plt.show()
print("Saved: assets/rubric_breakdown.png")

## Cell 12: Baseline vs Trained Comparison

In [ ]:
import sys
sys.path.insert(0, '.')

from inference import compare_baseline_vs_trained

baseline_rewards, trained_rewards = compare_baseline_vs_trained(
    trained_model_name=OUTPUT_MODEL,
    n_episodes=10
)

# Bar chart comparison
plt.figure(figsize=(8, 4))
x = range(len(baseline_rewards))
plt.bar(
    [i - 0.2 for i in x], baseline_rewards, 0.35,
    label="Random baseline", color="#FF6B6B", alpha=0.8
)
plt.bar(
    [i + 0.2 for i in x], trained_rewards, 0.35,
    label="Trained Butler", color="#6C63FF", alpha=0.8
)
plt.xlabel("Episode")
plt.ylabel("Reward (0–1)")
plt.title("Baseline vs Trained Butler — 10 Eval Episodes")
plt.legend()
plt.tight_layout()
plt.savefig("assets/baseline_vs_trained.png", dpi=150)
plt.show()
print("Saved: assets/baseline_vs_trained.png")

## Cell 13: Push to HF Hub

In [ ]:
trainer.push_to_hub(OUTPUT_MODEL)
print(f"\n✅ Model pushed to: https://huggingface.co/{OUTPUT_MODEL}")
print("\nTraining complete! 🎉")
print("\nNext steps:")
print(f"  1. Visit https://huggingface.co/{OUTPUT_MODEL}")
print(f"  2. Run inference: python inference.py --model {OUTPUT_MODEL} --todo 'Your task here'")
print(f"  3. Compare:       python inference.py --model {OUTPUT_MODEL} --compare")